# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eminamandzukic/FlyRank_starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Rule idea

For my Refresh/Content Opportunity Scoring lane, I want to prioritize content that already has meaningful search visibility but may have room for improvement in its search position.

Before using these signals in the rule, I will check whether:

1. Higher-impression content shows enough volume to make prioritization meaningful.
2. Search position behaves differently across visibility levels, so that position can help separate stronger and weaker opportunities.

The two signals I will audit are therefore:

- **Search impressions** — linked to FlyRank's volume 
- **Average search position** — used as an indicator of how strongly the content currently ranks.

I will only use information available before the decision point and will not use the future decline label or any label-derived fields.

### Signal check 1 — impressions

I first check whether higher-impression content forms a meaningful volume signal for prioritization. I bucket content items by March 1–15 impressions and compare how many items fall into each bucket.

This signal is linked to FlyRank's volume / quick-win logic.

In [8]:
import getpass
import duckdb
from huggingface_hub import hf_hub_download

HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token: ").strip()

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN
)

con = duckdb.connect()

model_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE WHEN report_date <= DATE '2026-03-15'
                 THEN gsc_impressions ELSE 0 END
        ) AS impressions_first15,

        SUM(
            CASE WHEN report_date <= DATE '2026-03-15'
                 THEN gsc_clicks ELSE 0 END
        ) AS clicks_first15,

        AVG(
            CASE WHEN report_date <= DATE '2026-03-15'
                 THEN gsc_avg_position END
        ) AS avg_position_first15

    FROM read_parquet('{march_path}')
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Rows:", len(model_frame))
model_frame.head()

Rows: 331437


,client_hash_id,content_hash_id,impressions_first15,clicks_first15,avg_position_first15
0,client_62f4a7e64f5e0096,content_92c381fbd361212e,233.0,0.0,4.807270
1,client_62f4a7e64f5e0096,content_22b90a28f54c7b8c,131.0,0.0,0.996244
2,client_62f4a7e64f5e0096,content_c8ab52ae1cc4689b,0.0,0.0,NaN
3,client_62f4a7e64f5e0096,content_7acf59f90e1aa5c0,3.0,0.0,5.000000
4,client_62f4a7e64f5e0096,content_9e194fda22523210,144.0,0.0,4.841600


In [9]:
import pandas as pd

signal1 = model_frame[
    ["client_hash_id", "content_hash_id", "impressions_first15"]
].copy()

signal1["impression_bucket"] = pd.cut(
    signal1["impressions_first15"],
    bins=[-1, 0, 50, 200, 1000, float("inf")],
    labels=[
        "0",
        "1-50",
        "51-200",
        "201-1000",
        "1000+"
    ]
)

impression_bucket_table = (
    signal1
    .groupby("impression_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("impressions_first15", "median")
    )
    .reset_index()
)

impression_bucket_table

,impression_bucket,n,median_impressions
0,0,179456,0.0
1,1-50,59848,8.0
2,51-200,30250,102.0
3,201-1000,34916,442.0
4,1000+,26967,2283.0


**Verdict: confirmed**

The impression buckets show that search visibility varies substantially across content items. A large number of pages have zero impressions, while 34,916 items have 201–1000 impressions and 26,967 have more than 1000 impressions in the first half of March.

This supports using impressions as a volume signal in the baseline. Pages with meaningful search visibility represent more actionable opportunities than pages with little or no observed demand.

### Signal check 2 — average search position

I next check whether search position provides useful separation among content items with observed search visibility. If position varies meaningfully across pages, it can help identify visible pages that may still have room for improvement.

In [10]:
position_signal = model_frame[
    model_frame["impressions_first15"] > 0
][
    ["content_hash_id", "avg_position_first15"]
].copy()

position_signal["position_bucket"] = pd.cut(
    position_signal["avg_position_first15"],
    bins=[0, 3, 10, 20, 50, float("inf")],
    labels=[
        "1-3",
        "4-10",
        "11-20",
        "21-50",
        "50+"
    ],
    include_lowest=True
)

position_bucket_table = (
    position_signal
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_position=("avg_position_first15", "median")
    )
    .reset_index()
)

position_bucket_table

,position_bucket,n,median_position
0,1-3,17557,2.037063
1,4-10,70000,6.000000
2,11-20,26913,13.745273
3,21-50,27741,30.015372
4,50+,9770,65.107435


**Verdict: confirmed**

Average search position varies substantially across content items with observed search visibility. Most visible pages are not concentrated in a single position range: 26,913 items fall in positions 11–20, 27,741 in positions 21–50, and 9,770 beyond position 50.

This supports using search position together with impressions in the baseline. Impressions indicate whether there is meaningful observed demand, while position helps distinguish pages that are already ranking strongly from pages that may still have room for improvement.

## 2. Build the ranked queue (writes the CSV)

### Baseline rule

I prioritize a content item when it has at least 200 impressions in March 1–15 but its average search position is worse than 10.

For qualifying pages, I use impressions as the score so that pages with more observed search demand are reviewed first.

- **Reason code:** `visible_but_not_top10`
- **Action label:** `review_for_refresh`

In [11]:
from pathlib import Path
import numpy as np

baseline = model_frame.copy()

# One transparent rule
qualifies = (
    (baseline["impressions_first15"] >= 200)
    & (baseline["avg_position_first15"] > 10)
)

# Score: higher-volume qualifying pages rank first
baseline["baseline_score"] = np.where(
    qualifies,
    baseline["impressions_first15"],
    0
)

# One reason code
baseline["reason_code"] = np.where(
    qualifies,
    "visible_but_not_top10",
    ""
)

# Action label
baseline["action_label"] = np.where(
    qualifies,
    "review_for_refresh",
    "monitor"
)

# Ranked queue
queue = (
    baseline
    .sort_values("baseline_score", ascending=False)
    .reset_index(drop=True)
)

queue["rank"] = queue.index + 1

queue = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "impressions_first15",
        "clicks_first15",
        "avg_position_first15",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
]

print("Total rows:", len(queue))
print("Pages flagged for review:", (queue["baseline_score"] > 0).sum())

queue.head(10)

Total rows: 331437
Pages flagged for review: 22767


,rank,client_hash_id,content_hash_id,impressions_first15,clicks_first15,avg_position_first15,baseline_score,reason_code,action_label
0,1,client_23a62021009f63c4,content_e8a52cf3d5988c07,143173.0,353.0,16.018687,143173.0,visible_but_not_top10,review_for_refresh
1,2,client_23a62021009f63c4,content_36e53e9c707674fc,109909.0,115.0,33.354423,109909.0,visible_but_not_top10,review_for_refresh
2,3,client_23a62021009f63c4,content_3df3f32f3fd58dea,84041.0,120.0,24.501281,84041.0,visible_but_not_top10,review_for_refresh
3,4,client_23a62021009f63c4,content_5e1c049f62e33b11,72940.0,108.0,18.233335,72940.0,visible_but_not_top10,review_for_refresh
4,5,client_20259bd6705d81d4,content_82e35c4845e6c391,70169.0,29.0,18.269589,70169.0,visible_but_not_top10,review_for_refresh
5,6,client_23a62021009f63c4,content_df47d1b976106de4,66342.0,83.0,25.221991,66342.0,visible_but_not_top10,review_for_refresh
6,7,client_23a62021009f63c4,content_559cdd76da9306de,62418.0,2.0,37.866104,62418.0,visible_but_not_top10,review_for_refresh
7,8,client_23a62021009f63c4,content_bdf60c86117079be,60277.0,8.0,31.046299,60277.0,visible_but_not_top10,review_for_refresh
8,9,client_20259bd6705d81d4,content_9fff53e827550f9d,56470.0,278.0,22.357092,56470.0,visible_but_not_top10,review_for_refresh
9,10,client_23a62021009f63c4,content_573804af4f4fa09f,52948.0,21.0,29.926699,52948.0,visible_but_not_top10,review_for_refresh


In [12]:
from pathlib import Path

output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

queue.to_csv(output_path, index=False)

print("Wrote:", output_path)
print("Rows written:", len(queue))

Wrote: ../outputs/baseline_action_score.csv
Rows written: 331437


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [13]:
top20 = queue.head(20).copy()

top20[
    [
        "rank",
        "content_hash_id",
        "impressions_first15",
        "clicks_first15",
        "avg_position_first15",
        "reason_code",
        "action_label"
    ]
]

,rank,content_hash_id,impressions_first15,clicks_first15,avg_position_first15,reason_code,action_label
0,1,content_e8a52cf3d5988c07,143173.0,353.0,16.018687,visible_but_not_top10,review_for_refresh
1,2,content_36e53e9c707674fc,109909.0,115.0,33.354423,visible_but_not_top10,review_for_refresh
2,3,content_3df3f32f3fd58dea,84041.0,120.0,24.501281,visible_but_not_top10,review_for_refresh
3,4,content_5e1c049f62e33b11,72940.0,108.0,18.233335,visible_but_not_top10,review_for_refresh
4,5,content_82e35c4845e6c391,70169.0,29.0,18.269589,visible_but_not_top10,review_for_refresh
5,6,content_df47d1b976106de4,66342.0,83.0,25.221991,visible_but_not_top10,review_for_refresh
6,7,content_559cdd76da9306de,62418.0,2.0,37.866104,visible_but_not_top10,review_for_refresh
7,8,content_bdf60c86117079be,60277.0,8.0,31.046299,visible_but_not_top10,review_for_refresh
8,9,content_9fff53e827550f9d,56470.0,278.0,22.357092,visible_but_not_top10,review_for_refresh
9,10,content_573804af4f4fa09f,52948.0,21.0,29.926699,visible_but_not_top10,review_for_refresh


### Top-20 review

1. Rank 1 — review for refresh; very high impressions with position 16.0. Wrong if competition, not content quality, explains the ranking.
2. Rank 2 — review for refresh; high impressions with position 33.4. Wrong if impressions come from broad, weak-intent queries.
3. Rank 3 — review for refresh; strong visibility with position 24.5. Wrong if important queries already rank well.
4. Rank 4 — review for refresh; high impressions and position 18.2. Wrong if authority is the main limitation.
5. Rank 5 — review for refresh; high impressions and position 18.3. Wrong if low clicks reflect query mismatch.
6. Rank 6 — review for refresh; high impressions and position 25.2. Wrong if visibility is spread across unrelated queries.
7. Rank 7 — review for refresh; high impressions but position 37.9. Wrong if the page is mostly shown for irrelevant queries.
8. Rank 8 — review for refresh; strong impressions and position 31.0. Wrong if the low click count means weak demand.
9. Rank 9 — review for refresh; strong impressions, 278 clicks, position 22.4. Wrong if key queries already perform well.
10. Rank 10 — review for refresh; high impressions and position 29.9. Wrong if visibility does not translate into useful traffic.
11. Rank 11 — review for refresh; high impressions and position 35.1. Wrong if the very low click count signals weak relevance.
12. Rank 12 — review for refresh; high impressions and position 30.4. Wrong if impressions come from low-value searches.
13. Rank 13 — review for refresh; high impressions and position 22.6. Wrong if competition, not freshness, explains the gap.
14. Rank 14 — review for refresh; high impressions and position 23.4. Wrong if current performance is already appropriate for the intent.
15. Rank 15 — review for refresh; high impressions and position 20.9. Wrong if authority is the main bottleneck.
16. Rank 16 — review for refresh; high impressions and position 25.6. Wrong if the traffic opportunity is too fragmented.
17. Rank 17 — review for refresh; high impressions and position 34.8. Wrong if the page is too far from page one for a refresh alone to help.
18. Rank 18 — review for refresh; high impressions but position 37.4. Wrong if the 5 clicks indicate mostly irrelevant exposure.
19. Rank 19 — review for refresh; high impressions, 95 clicks, position 27.4. Wrong if valuable queries already rank well.
20. Rank 20 — review for refresh; high impressions and position 35.8. Wrong if the low click count means limited useful demand.

## 4. Weak picks + leakage check

### Weak picks

Some of the highest-ranked pages have very high impressions but very few clicks. For example, several top-ranked items receive only a handful of clicks despite tens of thousands of impressions.

This suggests a weakness in the baseline: ranking only by impressions after the threshold can push pages with weak query relevance or weak click demand too high in the queue.

### Leakage check

The baseline uses only March 1–15 search signals that would be available at the decision moment.

It does not use the future March 16–31 outcome window, `is_declining`, or any label-derived field. Therefore the ranked queue does not contain future-window leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.